# NACC data importation, analysis and preparation

Created by: Sahana Kowshik

## Libraries and Data Files
- Import necessary libraries, `pandas` and `numpy`.
- Load several CSV files containing data and metadata for analysis.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import json
import string

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"

In [3]:
all_nacc = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_vnum_unique_nacc65.csv')
nacc_variable = pd.read_csv(f"{directory_path}/nacc_variable.csv")
nacc_allowable_code = pd.read_csv(f"{directory_path}/nacc_allowable_code.csv")

In [4]:
replacements = {
    '.F': np.nan, '0': np.nan, '0.0': np.nan, 
    '8': np.nan, '8.0': np.nan, 'SPECT': np.nan, 
    'spect': np.nan, 'CT': np.nan, '3/3': np.nan,
    'HMPAO-SPECT': np.nan, 'SPECT-HMPAO': np.nan, 
    'BRAIN SPECT': np.nan, 'brain spect': np.nan, 
    'CT BRAIN': np.nan, 'sMRI; outside SPECT scan': np.nan, 
    'Technetium Neurolite Perfusion': np.nan, '(Spect) 2004': np.nan, 
    'SPECT 2017 NON SPECIFIC': np.nan, 'Proton spectroscopy MR': np.nan, 
    'SMRI': np.nan, 'Spect': np.nan, 'SPECT-ECD': np.nan, 
    'Spect-ECD': np.nan, 'HMPAO SPECT': np.nan, 'EDC SPECT': np.nan, 
    'ECD SPECT': np.nan, 'HMPAD-SPECT:HYPERFUSION': np.nan, 'Technetium Neurolite Perfusion (Spect) 2004': np.nan, 'LT 2-25-2015': np.nan, 'APO E 3/3': np.nan, 'APO E 4/4': np.nan
}

# Replace the values in the DataFrame
all_nacc = all_nacc.replace({'-4': np.NaN, -4: np.NaN, '-4.0': np.NaN, -4.0: np.NaN})
all_nacc['OTHBIOMX'] = all_nacc['OTHBIOMX'].replace(replacements)
all_nacc.loc[all_nacc['OTHBIOMX'].str.contains('apoe', case=False, na=False), 'OTHBIOMX'] = np.nan

In [5]:
all_nacc.drop(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM', 'NACCC1', 'NACCC2', 'NEWINF','NACCAGEB', 'WHODIDDX', 'DXMETHOD'], axis=1, inplace=True)

In [6]:
# all_nacc[~all_nacc['OTHBIOMX'].isna()]['OTHBIOMX'].value_counts().to_csv("OTHBIOMX_check.csv")

## Data Cleaning

- Fill missing values in variable_id column using forward filling method.

In [7]:
nacc_allowable_code['variable_id_conv'] = nacc_allowable_code['variable_id'].fillna(method='ffill')
nacc_allowable_code.rename(columns={"descriptor": "individual_descriptor"}, inplace=True)

## Data Merging

- Merge `nacc_variable` with `nacc_allowable_code` on variable IDs to consolidate variable descriptions and their corresponding codes.
- Filter merged data for specific UDS forms ('A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C1', 'C1, C2', 'C1, C2, C2T', 'C1,C2', 'C2, C2T', 'C2', 'C2T', 'G1', 'D2) to get clinical features.
- Filter merged data for specific UDS forms (B9 and D1) to get diagnosis features.

In [8]:
form = {
    'A1': 'Subject Demographics',
    'A2': 'Co-participant Demographics',
    'A3': 'Subject Family History',
    'A4': 'Subject Medications',
    'A5': 'Subject Health History',
    'B1': 'Physical',
    'B2': 'His and CVD',
    'B3': "Unified Parkinson's Disease Rating Scale (UPDRS)",
    'B4': 'Clinical Dementia Rating (CDR)',
    'B5': 'Neuropsychiatric Inventory Questionnaire (NPI-Q)',
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Functional Assessment Scale (FAS)',
    'B8': 'Physical/Neurological Exam Findings',
    'B9': 'Clinician Judgment of Symptoms',
    'C': 'Neuropsychological battery Summary Scores',
    'D1': 'Clinician Diagnosis',
    'D2': 'Clinician-assessed Medical Conditions',
    'G1': 'Genetic testing',
    'I': 'Imaging evidence',
    'CSF': 'CSF evidence'
}

In [9]:
PET = ['AMYLPET', 'FDGAD', 'TAUPETAD', 'FDGFTLD', 'TPETFTLD', 'DATSCAN', 'OTHBIOMX']
CSF = ['AMYLCSF', 'CSFTAU']
MRI = ['CVDIMAG1', 'CVDIMAG2', 'CVDIMAGX', 'IMAGLINF', 'IMAGLAC', 'IMAGMACH', 'IMAGMICH', 'IMAGMWMH', 'IMAGEWMH', 'INFNETW', 'INFWMH', 'MRFTLD', 'HIPPATR']
IMAGING = MRI + PET

imag_fea = MRI + PET + CSF

In [10]:
IMAGING

['CVDIMAG1',
 'CVDIMAG2',
 'CVDIMAGX',
 'IMAGLINF',
 'IMAGLAC',
 'IMAGMACH',
 'IMAGMICH',
 'IMAGMWMH',
 'IMAGEWMH',
 'INFNETW',
 'INFWMH',
 'MRFTLD',
 'HIPPATR',
 'AMYLPET',
 'FDGAD',
 'TAUPETAD',
 'FDGFTLD',
 'TPETFTLD',
 'DATSCAN',
 'OTHBIOMX']

In [11]:
set(nacc_variable['id']) - set(nacc_allowable_code['variable_id_conv'])

set()

In [12]:
set(nacc_allowable_code['variable_id_conv']) - set(nacc_variable['id'])

set()

In [13]:
merged_df = pd.merge(nacc_variable, nacc_allowable_code, left_on='id', right_on='variable_id_conv', how='outer')
merged_df.loc[merged_df['id'].isin(IMAGING), 'form_id'] = 'I'
merged_df.loc[merged_df['id'].isin(CSF), 'form_id'] = 'CSF'
form_id_desc = []
for i in list(merged_df['form_id']):
    try:
        form_id_desc.append(form[i])
    except:
        form_id_desc.append(np.NaN)
merged_df['form_id_desc'] = form_id_desc
merged_df = merged_df.sort_values('form_id').reset_index(drop=True)

forms = ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C', 'G1', 'D2']
merged_df_patient = merged_df[(merged_df['form_id'].isin(forms)) | (merged_df['variable_id_conv'].isin(imag_fea))].reset_index(drop=True)
merged_df_diag = merged_df[(merged_df['form_id'].isin(['B9', 'D1'])) & (~merged_df['variable_id_conv'].isin(imag_fea))].reset_index(drop=True)


In [14]:
merged_df_patient

,id,form_id,variable_type_id,source,descriptor,variable_id,code_1,code_2,individual_descriptor,variable_id_conv,form_id_desc
0,NACCLIVS,A1,D,v1-3,Living situation,NaN,3,NaN,Lives with relative or friend,NACCLIVS,Subject Demographics
1,PRIMLANG,A1,O,v1-3,Primary language,NaN,9,NaN,Unknown,PRIMLANG,Subject Demographics
2,PRIMLANX,A1,O,v1-3,"Primary language, other- specify",PRIMLANX,<UND>,NaN,Any text or numbers,PRIMLANX,Subject Demographics
3,PRIMLANX,A1,O,v1-3,"Primary language, other- specify",NaN,-4,NaN,Not available: UDS form submitted did not coll...,PRIMLANX,Subject Demographics
4,PRIMLANX,A1,O,v1-3,"Primary language, other- specify",NaN,-4,NaN,Not available: UDS form submitted did not coll...,PRIMLANX,Subject Demographics
...,...,...,...,...,...,...,...,...,...,...,...
2906,AMYLPET,I,O,v3,Abnormally elevated amyloid on PET,NaN,-4,NaN,Not available: UDS form submitted did not coll...,AMYLPET,Imaging evidence
2907,FDGAD,I,O,v3,FDG-PET pattern of AD,NaN,8,NaN,Unknown/not assessed,FDGAD,Imaging evidence
2908,FDGAD,I,O,v3,FDG-PET pattern of AD,NaN,-4,NaN,Not available: UDS form submitted did not coll...,FDGAD,Imaging evidence
2909,FDGAD,I,O,v3,FDG-PET pattern of AD,NaN,-4,NaN,Not available: UDS form submitted did not coll...,FDGAD,Imaging evidence


## Data Transformation

- Generate dictionaries for 
    - variable name and text diagnosis
    - descriptor codes and text descriptions for each code.

In [15]:
patient_dict = dict(zip(merged_df_patient['id'], zip(merged_df_patient['form_id_desc'], merged_df_patient['descriptor'])))
# patient_dict = dict(zip(merged_df_patient['id'], merged_df_patient['descriptor']))
patient_code_dict = {
    id_value: {
        (row['code_1'] if pd.isna(row['code_2']) else "range"):
        (row['individual_descriptor'] if pd.isna(row['code_2']) else (int(row['code_1']), int(row['code_2'])))
        for index, row in group.iterrows()
    }
    for id_value, group in merged_df_patient.groupby('id')
}

In [16]:
diag_dict = dict(zip(merged_df_diag['id'], zip(merged_df_diag['form_id_desc'], merged_df_diag['descriptor'])))
# diag_dict = dict(zip(merged_df_diag['id'], merged_df_diag['descriptor']))
diag_code_dict = {
    id_value: {
        (row['code_1'] if pd.isna(row['code_2']) else "range"):
        (row['individual_descriptor'] if pd.isna(row['code_2']) else (int(row['code_1']), int(row['code_2'])))
        for index, row in group.iterrows()
    }
    for id_value, group in merged_df_diag.groupby('id')
}

In [17]:
patient_dict['HIPPATR']

('Imaging evidence', 'Hippocampal atrophy')

In [18]:
patient_code_dict['HIPPATR']

{'-4': 'Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question',
 '8': 'Unknown/not assessed',
 '1': 'Present',
 '0': 'Absent'}

In [19]:
diag_dict['COGFLAGO']

('Clinician Judgment of Symptoms',
 'At what age did the fluctuating cognition begin?')

In [20]:
diag_code_dict['COGFLAGO']

{'-4': 'Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question',
 'range': (15, 110),
 '888': 'Not applicable, no or unknown fluctuating cognition',
 '999': 'Age unknown'}

In [21]:
def remove_non_printable_chars(text):
    """Removes non-printable characters from a string."""
    # Create a set of printable characters
    printable_chars = set(string.printable)

    # Filter the string to keep only printable characters
    return ''.join(filter(lambda x: x in printable_chars, text))

In [22]:
def get_json(summary):
    json_file = json.loads(remove_non_printable_chars(summary.replace('```json\n', '').replace('\n```', '').replace(",\n\t}", "\n\t}").replace(",\n}", "\n}").replace("\t", "    ").replace("\\", '')))
    return json.dumps(json_file, indent=4)

## Patient Record Dictionary Creation

- Filter the record columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding record texts.

In [23]:
patient_dict.update({f"DRUG{i}" : patient_dict["DRUG1-DRUG40"] for i in range(1,41)})
patient_dict = {k: v for k, v in patient_dict.items() if k in all_nacc.columns}
filtered_nacc = all_nacc[list(patient_dict.keys()) + ["NACCID"]].replace({-4: np.NaN, "-4": np.NaN})
remove_cols = set(filtered_nacc.columns).intersection(set(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM', 'NACCC1', 'NEWINF','NACCAGEB']))
filtered_nacc.drop(remove_cols, inplace=True, axis=1)
remove_cols

set()

In [24]:
import pandas as pd
import json
from tqdm import tqdm

# Assuming you have the necessary data and dictionaries: patient_dict, patient_code_dict
# Ensure the folder for storing JSON files exists or is created beforehand
patient_texts = []
for i, row in tqdm(filtered_nacc.iterrows(), total=len(filtered_nacc)):
    patient_data = {}
    for k, v in dict(row).items():
        if k not in row or k == "NACCID":
            continue
        
        # if (k in patient_code_dict) and ("range" in patient_code_dict[k]) and ("Demographics" not in patient_dict[k][0]):
        #     val = patient_dict[k][0] + patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')'
        # else:
        #     val = patient_dict[k][0] + patient_dict[k][1]
        
        if patient_dict[k][0] not in patient_data:
            patient_data[patient_dict[k][0]] = {}
            
        if (k in patient_code_dict) and ("range" in patient_code_dict[k]) and ("Demographics" not in patient_dict[k][0]):
            val = patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')'
        else:
            val = patient_dict[k][1]
        
        if not pd.isna(v):
            if not isinstance(v, str):
                if str(int(v)) in patient_code_dict[k]:
                    try:
                        if ('unknown' in patient_code_dict[k][str(int(v))].lower()) or ('not applicable' in patient_code_dict[k][str(int(v))].lower()) or ('not available' in patient_code_dict[k][str(int(v))].lower()) or ('did not report' in patient_code_dict[k][str(int(v))].lower()) or ('not assessed' in patient_code_dict[k][str(int(v))].lower()):
                            continue
                    except:
                        print(k, v, patient_code_dict[k][str(int(v))])
                    patient_data[patient_dict[k][0]][val.replace(';', ',')] = patient_code_dict[k][str(int(v))].replace(';', ',')
                elif "range" in patient_code_dict[k] and int(v) in range(patient_code_dict[k]["range"][0], patient_code_dict[k]["range"][1]+1):
                    if isinstance(v, str) and (('unknown' in v.lower()) or ('not applicable' in v.lower()) or ('not available' in v.lower()) or ('did not report' in v.lower())) or ('not assessed' in patient_code_dict[k][str(int(v))].lower()):
                        continue
                    patient_data[patient_dict[k][0]][val.replace(';', ',')] = v
            else:
                patient_data[patient_dict[k][0]][val.replace(';', ',')] = v
                
    patient_data_str = "\n{\n"
    for key, value in patient_data.items():
        if len(value) == 0:
            continue
        patient_data_str += f'\t"{key}"' + ': {\n'
        for k, v in value.items():
            if isinstance(v, str):
                patient_data_str += f'\t\t"{k}": "{v}",\n'
            else:
                patient_data_str += f'\t\t"{k}": {v},\n'
        patient_data_str += '\t},\n'
    patient_data_str += '}\n'
    patient_texts.append(get_json(patient_data_str))
    # # Write to a JSON file
    # with open(f'patient_data_{i}.json', 'w') as f:
    #     json.dump(patient_data, f, ensure_ascii=False, indent=4)
    
    # break


100%|██████████| 188700/188700 [17:30<00:00, 179.64it/s]


In [25]:
all_nacc.iloc[1000]['NACCNE4S']

9

In [39]:
for text in patient_texts:
    if "no assessed" in text.lower():
        print(text)
        break

In [ ]:
print(patient_texts[0])

In [27]:
print(json.loads(patient_texts[0]))

{'Subject Demographics': {'Living situation': 'Lives with spouse or partner', 'Primary language': 'English', 'Level of independence': 'Able to live independently', 'Race': 'White', 'Second race': 'None reported', 'Hispanic/Latino ethnicity': 'No', 'Primary reason for coming to ADC': 'To participate in a research study', 'Principal referral source': 'Other', 'Is the subject left- or right-handed?': 'Right-handed', 'Type of residence': 'Single- or multi-family private residence (apartment, condo, house)', "Subject's age at visit": 62, 'Marital status': 'Married', 'Years of education': 16, "Subject's sex": 'Female', "Subject's month of birth": 2, "Subject's year of birth": 1944, 'Third race': 'None reported', 'Derived NIH race definitions': 'White'}, 'Co-participant Demographics': {"Co-participant's relationship to subject": 'Friend, neighbor, or someone known through family, friends, work, or community', 'Derived NIH race definitions': 'White', "Co-participant's year of birth": 1930.0, "

In [28]:
# from tqdm import tqdm

# patient_texts = []
# for i, row, in tqdm(filtered_nacc.iterrows()):
#     st = ""
#     for k, v in dict(row).items():
#         if k not in row:
#             continue
#         if k == "NACCID":
#             continue
        
#         if (k in patient_code_dict) and ("range" in patient_code_dict[k]) and ("Demographics" not in patient_dict[k][0]):
#             # print(patient_dict[k][0] + patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')')
#             # continue
#             val = patient_dict[k][0] + patient_dict[k][1] + " (range " + str(patient_code_dict[k]["range"]).replace(',', ' -') + ')'
#         else:
#             val = patient_dict[k][0] + patient_dict[k][1]
        
#         if not pd.isna(v):
#             if not isinstance(v, str):
#                 if str(int(v)) in patient_code_dict[k]:
#                     try:
#                         if ('unknown' in patient_code_dict[k][str(int(v))].lower()) | ('not applicable' in patient_code_dict[k][str(int(v))].lower()) | ('not available' in patient_code_dict[k][str(int(v))].lower()) | ('did not report' in patient_code_dict[k][str(int(v))].lower()):
#                             continue
#                     except:
#                         print(k, v, patient_code_dict[k][str(int(v))])
#                     st += f"{val.replace(';', ',')}: {patient_code_dict[k][str(int(v))].replace(';', ',')}; "
#                 elif "range" in patient_code_dict[k] and int(v) in range(patient_code_dict[k]["range"][0], patient_code_dict[k]["range"][1]+1):
#                     if isinstance(v, str) and (('unknown' in v.lower()) | ('not applicable' in v.lower()) | ('not available' in v.lower()) | ('did not report' in v.lower())):
#                         continue
#                     st += f"{val.replace(';', ',')}: {v}; "
#             else:
#                 st += f"{val.replace(';', ',')}: {v}; "
        
#     patient_texts.append(st)
#     # break

In [29]:
len(patient_texts)

188700

## Diagnosis Dictionary Creation

- Filter the diagnostic columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding diagnostic texts.

In [30]:
from tqdm import tqdm
avail_keys = set(all_nacc.columns).intersection(set(list(diag_dict.keys()) + ['NACCID']))
filtered_diag_nacc = all_nacc[list(avail_keys)].replace({-4: np.NaN, "-4": np.NaN})
remove_cols = set(filtered_diag_nacc.columns).intersection(set(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM'] + imag_fea))
filtered_diag_nacc.drop(remove_cols, inplace=True, axis=1)

In [31]:
remove_cols

set()

In [32]:
# import pandas as pd
# import json
# from tqdm import tqdm

# # Make sure to define or import your diag_dict and diag_code_dict appropriately
# # Ensure the folder for storing JSON files exists or is created beforehand
# diag_texts = []
# for i, row in tqdm(filtered_diag_nacc.iterrows(), total=len(filtered_diag_nacc)):
#     diag_data = {}
#     for k, v in dict(row).items():
#         if k == "NACCID":
#             continue
        
#         val = diag_dict[k][1]# + diag_dict[k][1]
        
#         if not pd.isna(v):
#             if not isinstance(v, str):
#                 if str(int(v)) in diag_code_dict[k]:
#                     try:
#                         if ('unknown' in diag_code_dict[k][str(int(v))].lower()) or ('not applicable' in diag_code_dict[k][str(int(v))].lower()) or ('not available' in diag_code_dict[k][str(int(v))].lower()) or ('did not report' in diag_code_dict[k][str(int(v))].lower()):
#                             continue
#                     except:
#                         print(k, v, diag_code_dict[k][str(int(v))])
#                     diag_data[val.replace(';', ',')] = diag_code_dict[k][str(int(v))].replace(';', ',')
#                 elif "range" in diag_code_dict[k] and int(v) in range(diag_code_dict[k]["range"][0], diag_code_dict[k]["range"][1]+1):
#                     if isinstance(v, str) and (('unknown' in v.lower()) or ('not applicable' in v.lower()) or ('not available' in v.lower()) or ('did not report' in v.lower())):
#                         continue
#                     diag_data[val.replace(';', ',')] = v
#             else:
#                 diag_data[val.replace(';', ',')] = v
    
#     diag_data_str = "```json\n{\n"
#     for key, value in diag_data.items():
#         if isinstance(value, str):
#             v = value.replace(' by clinician', '').replace(" (D1 form)", '')
#             diag_data_str += f'\t"{key}": "{v}",\n'
#         else:
#             diag_data_str += f'\t"{key}": {value},\n'
        
#     diag_data_str += '}\n```'
#     diag_texts.append(get_json(diag_data_str))
#     break


In [33]:
import pandas as pd
import json
from tqdm import tqdm

# Assuming you have the necessary data and dictionaries: diag_dict, diag_code_dict
# Ensure the folder for storing JSON files exists or is created beforehand
diag_texts = []
for i, row in tqdm(filtered_diag_nacc.iterrows(), total=len(filtered_diag_nacc)):
    diag_data = {}
    for k, v in dict(row).items():
        if k not in row or k == "NACCID":
            continue
        
        # if (k in diag_code_dict) and ("range" in diag_code_dict[k]) and ("Demographics" not in diag_dict[k][0]):
        #     val = diag_dict[k][0] + diag_dict[k][1] + " (range " + str(diag_code_dict[k]["range"]).replace(',', ' -') + ')'
        # else:
        #     val = diag_dict[k][0] + diag_dict[k][1]
        
        if diag_dict[k][0] not in diag_data:
            diag_data[diag_dict[k][0]] = {}
            
        if (k in diag_code_dict) and ("range" in diag_code_dict[k]) and ("Demographics" not in diag_dict[k][0]):
            val = diag_dict[k][1] + " (range " + str(diag_code_dict[k]["range"]).replace(',', ' -') + ')'
        else:
            val = diag_dict[k][1]
        
        if not pd.isna(v):
            if not isinstance(v, str):
                if str(int(v)) in diag_code_dict[k]:
                    try:
                        if ('unknown' in diag_code_dict[k][str(int(v))].lower()) or ('not applicable' in diag_code_dict[k][str(int(v))].lower()) or ('not available' in diag_code_dict[k][str(int(v))].lower()) or ('did not report' in diag_code_dict[k][str(int(v))].lower()) or ('not assessed' in diag_code_dict[k][str(int(v))].lower()):
                            continue
                    except:
                        print(k, v, diag_code_dict[k][str(int(v))])
                    diag_data[diag_dict[k][0]][val.replace(';', ',')] = diag_code_dict[k][str(int(v))].replace(';', ',')
                elif "range" in diag_code_dict[k] and int(v) in range(diag_code_dict[k]["range"][0], diag_code_dict[k]["range"][1]+1):
                    if isinstance(v, str) and (('unknown' in v.lower()) or ('not applicable' in v.lower()) or ('not available' in v.lower()) or ('did not report' in v.lower())) or ('not assessed' in patient_code_dict[k][str(int(v))].lower()):
                        continue
                    diag_data[diag_dict[k][0]][val.replace(';', ',')] = v
            else:
                diag_data[diag_dict[k][0]][val.replace(';', ',')] = v
                
    diag_data_str = "\n{\n"
    for key, value in diag_data.items():
        if len(value) == 0:
            continue
        diag_data_str += f'\t"{key}"' + ': {\n'
        for k, v in value.items():
            if isinstance(v, str):
                diag_data_str += f'\t\t"{k}": "{v}",\n'
            else:
                diag_data_str += f'\t\t"{k}": {v},\n'
        diag_data_str += '\t},\n'
    diag_data_str += '}\n'
    diag_texts.append(get_json(diag_data_str))
    # # Write to a JSON file
    # with open(f'diag_data_{i}.json', 'w') as f:
    #     json.dump(diag_data, f, ensure_ascii=False, indent=4)
    
    # break


100%|██████████| 188700/188700 [09:26<00:00, 333.03it/s]


In [58]:
# diag_texts = []
# for i, row, in tqdm(filtered_diag_nacc.iterrows()):
#     st = ""
#     for k, v in dict(row).items():
#         if k == "NACCID":
#             continue
#         val = diag_dict[k][0] + diag_dict[k][1]
                
#         if not pd.isna(v):
#             if not isinstance(v, str):
#                 if str(int(v)) in diag_code_dict[k]:
#                     try:
#                         if ('unknown' in diag_code_dict[k][str(int(v))].lower()) | ('not applicable' in diag_code_dict[k][str(int(v))].lower()) | ('not available' in diag_code_dict[k][str(int(v))].lower()) | ('did not report' in diag_code_dict[k][str(int(v))].lower()):
#                             continue
#                     except:
#                         print(k, v, diag_code_dict[k][str(int(v))])
#                     st += f"{val.replace(';', ',')}: {diag_code_dict[k][str(int(v))].replace(';', ',')}; "
#                 elif "range" in diag_code_dict[k] and int(v) in range(diag_code_dict[k]["range"][0], diag_code_dict[k]["range"][1]+1):
#                     if isinstance(v, str) and (('unknown' in v.lower()) | ('not applicable' in v.lower()) | ('not available' in v.lower()) | ('did not report' in v.lower())):
#                         continue
#                     st += f"{val.replace(';', ',')}: {v}; "
#             else:
#                 st += f"{val.replace(';', ',')}: {v}; "
        
#     diag_texts.append(st)

In [40]:
for text in diag_texts:
    if "no assessed" in text.lower():
        print(text)
        break

In [35]:
print(json.loads(diag_texts[0]))

{'Clinician Judgment of Symptoms': {'Subject currently manifests meaningful change in behavior - Psychosis - Abnormal, false, or delusional beliefs': 'No', 'Subject currently manifests meaningful change in behavior - Apathy, withdrawal': 'No', 'Subject currently manifests meaningful change in behavior - Psychosis - Visual hallucinations': 'No', 'Indicate whether the subject currently is meaningfully impaired, relative to previously attained abilities, in visuospatial function': 'No', 'Indicate whether the subject currently is meaningfully impaired, relative to previously attained abilities, in attention or concentration': 'No', 'Indicate whether the subject currently is meaningfully impaired, relative to previously attained abilities, in memory': 'Yes', 'Indicate whether the subject currently is meaningfully impaired, relative to previously attained abilities, in language': 'Yes', 'Does the subject report a decline in memory (relative to previously attained abilities)?': 'Yes', 'Overal

In [36]:
len(diag_texts)

188700

## Data Export

- Save the diagnosis dictionary to a JSON file for further use.

In [37]:
all_nacc["diag_summary"] = diag_texts
all_nacc["patient_summary"] = patient_texts

In [38]:
import os
if not os.path.exists(f"{directory_path}/training_data"):
    os.makedirs(f"{directory_path}/training_data", exist_ok=True)

all_nacc.to_csv(f"{directory_path}/training_data/NACC_wjson.csv", index=False)